# Paper 5, RB-1b - cuDNN version check

**Naming.** W1-W7 = September runs behind the paper. RB = November rebuttal runs. RB-0 = the single-head
run (files `w8_*`). RB-1 = the drift check. This is **RB-1b**, a short follow-up to RB-1; it writes to
`./rb1b_output/`.

**Why this run, before RB-2.** RB-1 showed the current machine is perfectly repeatable and that our images,
weights and free GPU memory are not why ViT-S's numbers moved (≤4.7e-4) since September. It also found a
concrete suspect: `env_paper5` contains two cuDNN libraries (`nvidia-cudnn-cu12` 9.10.2, the version
PyTorch 2.10 is built for, and `nvidia-cudnn-cu13` 9.26), and PyTorch loads **9.26**. September's env almost
certainly ran 9.10.2. Today ViT-S's first layer runs in exact fp32, while September's did not, which is
what a different cuDNN kernel choice would do. RB-1b tests this directly: it re-measures the same three
conditions (baseline, floor, Cref) with **cuDNN 9.10.2** in a clean copy of the environment, with
`env_paper5` (9.26) as the control. If 9.10.2 brings back September, we can run every remaining rebuttal
experiment on the paper's exact numerics.

**Fixes from RB-1 (my errors there).** (1) Workers now exit with `os._exit(0)` after saving, and "completed"
is judged by the saved result, not the exit code, so the harmless shutdown crash no longer shows as FAIL.
(2) The first-layer check now uses the real batch of 256 images that the measurement uses (RB-1 used a
batch of 2, for which cuDNN can pick a different kernel), and compares it with a float64 computation of
the same batch inside the same process. (3) The verdict rules no longer infer TF32 use from today's ViT-S.

| Setting | Python environment | Models | Purpose |
|---|---|---|---|
| `cudnn926_default` | `env_paper5` (cuDNN 9.26) | all five | control: must equal RB-0 bit for bit |
| `cudnn910_default_a` | clean copy (cuDNN 9.10.2) | all five | does 9.10.2 bring back September? |
| `cudnn910_default_b` | clean copy | ViT-S, naver | is the clean copy repeatable? |
| `cudnn910_tf32_off` | clean copy | ViT-S, naver | does September's ViT-S depend on TF32 under 9.10.2? |

Runtime ~15-20 min on one H100 (plus building the clean copy, see the instructions doc).

In [ ]:
import os, sys, json, time, shutil, subprocess, re, traceback, platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
try:
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"]); import pandas as pd

# config
REPO_PATH = os.environ.get("RB1B_REPO", "")
PY_926    = os.environ.get("RB1B_PY_926", "")     # python of env_paper5 (cuDNN 9.26)
PY_910    = os.environ.get("RB1B_PY_910", "")     # python of the clean copy (cuDNN 9.10.2)
W8_DIR    = os.environ.get("RB1B_W8_DIR", "")     # RB-0 folder, for checking the typed anchors
OUT_DIR   = Path(os.environ.get("RB1B_OUT", "rb1b_output"))
TEST_MODE = os.environ.get("RB1B_TEST", "0") == "1"
ONLY      = [c for c in os.environ.get("RB1B_ONLY", "").split(",") if c]

N_IMAGES, BATCH_SIZE, TORCH_SEED, METRIC, INPUT_SIZE = 1000, 256, 20260908, "manhattan", 224   # W5, unchanged
SSDC_DATASET = "benjamin-paine/imagenet-1k-256x256"
WINDOW = list(range(0, 6))
MODEL_NAMES = {"naver":    "vit_base_patch16_rope_224.naver_in1k",
               "DINOv3":   "vit_base_patch16_dinov3.lvd1689m",
               "small":    "vit_small_patch16_rope_224.naver_in1k",
               "rope_ape": "vit_base_patch16_rope_ape_224.naver_in1k",
               "reg1_gap": "vit_base_patch16_rope_reg1_gap_256.sbb_in1k"}
ALL5 = ["naver", "DINOv3", "small", "rope_ape", "reg1_gap"]          # W5's load order
CONDITIONS = ["baseline", "floor", "Cref"]
EXPECT = {"910": (9, 10), "926": (9, 26)}                           # (major, minor) of the cuDNN that must load
SETTINGS = [  # name, env, flags, models
    ("cudnn926_default",   "926", {},                          ALL5),
    ("cudnn910_default_a", "910", {},                          ALL5),
    ("cudnn910_default_b", "910", {},                          ["small", "naver"]),
    ("cudnn910_tf32_off",  "910", {"cudnn_allow_tf32": False}, ["small", "naver"]),
]
if TEST_MODE:
    N_IMAGES, BATCH_SIZE = 24, 8
    OUT_DIR = Path(os.environ.get("RB1B_OUT", "rb1b_test_output"))
OUT_DIR.mkdir(exist_ok=True, parents=True)
PYTHONS = {"926": PY_926 or sys.executable, "910": PY_910 or sys.executable}

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H-%M-%SZ")
GATES = []
GATE_FILE = OUT_DIR / f"rb1b_gates_session_{RUN_STAMP}.json"
def gate(name, scope, passed, detail="", skip=False):
    status = "SKIP" if skip else ("PASS" if passed else "FAIL")
    GATES.append({"gate": name, "scope": scope, "status": status, "detail": detail, "session": RUN_STAMP})
    print(f"  [{status}] {scope}: {name}" + (f": {detail}" if detail else ""), flush=True)
    GATE_FILE.write_text(json.dumps(GATES, indent=1))
    return passed
print("pythons:", PYTHONS)
print(f"images {N_IMAGES} | batch {BATCH_SIZE} | test mode {TEST_MODE} | output {OUT_DIR.resolve()} | run {RUN_STAMP}")
if not TEST_MODE and (not PY_910 or not PY_926):
    raise SystemExit("Set RB1B_PY_910 and RB1B_PY_926 to the two environments' python executables (see the instructions doc).")

In [ ]:
def resolve_repo():
    cands = [Path(REPO_PATH).expanduser()] if REPO_PATH else []
    here = Path.cwd()
    for base in [here, *list(here.parents)[:4]]:
        cands += [base / "vit-sae-analysis" / "project_code" / "src", base / "project_code" / "src", base / "src", base]
    for c in cands:
        c = c.resolve()
        if (c / "metrics").exists() and (c / "main").exists() and (c / "experiments").exists():
            return c
    raise FileNotFoundError("No checkout with metrics/, main/, experiments/. Set RB1B_REPO.")
repo = resolve_repo()
print("repo root:", repo)
def sh(cmd, timeout=120):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return (r.stdout or "") + (r.stderr or "")
    except Exception as ex:
        return f"(failed: {ex!r})"
REPO_COMMIT = sh(f"git -C {repo} rev-parse HEAD").strip()
DRIVER = sh("nvidia-smi --query-gpu=index,name,uuid,driver_version,memory.used,memory.total --format=csv")
print("repo commit:", REPO_COMMIT); print(DRIVER)

## Pre-flight: which cuDNN does each environment really load?
A tiny GPU convolution in each environment, then read the library path from the process's own memory map.
Nothing is measured unless the clean copy loads cuDNN 9.10 and `env_paper5` loads 9.26, and both have the
same torch, timm and image-library versions.

In [ ]:
PREFLIGHT = r'''
import json, sys, importlib.metadata as md
import torch, torch.nn.functional as F
out = {"python": sys.executable, "torch": torch.__version__, "cudnn_reported": torch.backends.cudnn.version()}
if torch.cuda.is_available():
    x = torch.randn(2, 3, 32, 32, device="cuda"); w = torch.randn(8, 3, 16, 16, device="cuda")
    F.conv2d(x, w, stride=16); torch.cuda.synchronize()
libs = sorted({l.split()[-1] for l in open("/proc/self/maps") if "libcudnn" in l and l.split()[-1].startswith("/")})
out["libcudnn_loaded"] = libs
pk = {}
for d in md.distributions():
    n = (d.metadata.get("Name") or "").lower()
    if n in ("timm", "pillow", "numpy", "scipy", "datasets", "torchvision", "huggingface-hub", "safetensors") or "cudnn" in n:
        pk[n] = d.version
out["packages"] = pk
print("PREFLIGHT" + json.dumps(out))
'''
(OUT_DIR / "rb1b_preflight.py").write_text(PREFLIGHT)
PRE = {}
for tag, py in PYTHONS.items():
    r = subprocess.run([py, str(OUT_DIR / "rb1b_preflight.py")], capture_output=True, text=True, timeout=600)
    line = [l for l in r.stdout.splitlines() if l.startswith("PREFLIGHT")]
    if not line:
        print(r.stdout[-2000:], r.stderr[-2000:]); gate("pre-flight ran", tag, False, f"exit {r.returncode}"); continue
    PRE[tag] = json.loads(line[0][len("PREFLIGHT"):])
    p = PRE[tag]
    print(f"\n[{tag}] {p['python']}\n   torch {p['torch']} | cuDNN reported {p['cudnn_reported']}\n   loaded: {p['libcudnn_loaded']}\n   packages: {p['packages']}")
    if not TEST_MODE:
        v = p["cudnn_reported"] or 0
        major, minor = v // 10000, (v // 100) % 100
        gate(f"environment {tag} loads cuDNN {EXPECT[tag][0]}.{EXPECT[tag][1]}", tag, (major, minor) == EXPECT[tag],
             f"reported {v}; loaded {[Path(x).parent.parent.parent.name + '/' + Path(x).name for x in p['libcudnn_loaded']]}")
if not TEST_MODE and len(PRE) == 2:
    a, b = PRE["910"], PRE["926"]
    gate("both environments run torch 2.10.0+cu128", "env", a["torch"] == b["torch"] == "2.10.0+cu128", f"{a['torch']} / {b['torch']}")
    same = {k: (a["packages"].get(k), b["packages"].get(k)) for k in ("timm", "pillow", "numpy", "scipy", "datasets", "torchvision", "huggingface-hub", "safetensors")}
    diff = {k: v for k, v in same.items() if v[0] != v[1]}
    gate("the two environments differ only in cuDNN (same timm, pillow, numpy, scipy, datasets, torchvision, hub, safetensors)",
         "env", not diff, str(diff) if diff else "identical")
(OUT_DIR / "rb1b_preflight.json").write_text(json.dumps({"preflight": PRE, "driver": DRIVER, "repo_commit": REPO_COMMIT}, indent=1))
PRE_OK = TEST_MODE or not [g for g in GATES if g["status"] == "FAIL"]
print("\npre-flight", "OK" if PRE_OK else "FAILED: fix the environments before measuring (see the instructions doc)")

## Anchors
September (W5) values come from the repo's committed files. RB-0 values are typed at full precision from
`w8_partA_*.json` and checked against the RB-0 folder when it is found.

In [ ]:
RES_ROOT = repo.parents[1] / "results" / "runs" / "imagenet1k_val" / "rope_interventions"
W5 = {}
for key in MODEL_NAMES:
    hits = sorted(RES_ROOT.glob(f"w5/w5_partA_{key}_*.json"))
    if not hits:
        gate("committed W5 file found", key, False, "missing"); continue
    c = json.load(open(hits[0]))["conditions"]
    W5[key] = {"baseline": c["baseline"]["ssdc"], "floor": c["floor"]["ssdc"], "Cref": c["Cref_prefix5"]["ssdc"]}
HEAD_W5 = {"small": (0.371434, 0.244217), "naver": (0.469422, 0.329663), "DINOv3": (0.948654, 0.442640),
           "rope_ape": (0.589777, 0.263854), "reg1_gap": (0.689023, 0.244575)}
for key, (b4, c11) in HEAD_W5.items():
    if key in W5:
        gate("committed W5 file matches its headline values", key,
             abs(W5[key]["baseline"][4] - b4) < 1e-6 and abs(W5[key]["Cref"][11] - c11) < 1e-6)

RB0 = {
    "small": {
        "baseline": [0.016578371328500142, 0.00395960172172849, 0.04124018025894116, 0.16859474094367033, 0.3713740002085806, 0.40662082251490533, 0.5489240691401012, 0.5064005532218341, 0.46002794920552376, 0.4585050970469657, 0.4455475282553677, 0.40882081445458496],
        "floor": [0.016578371328500142, 0.01519492949780967, 0.008314211653128038, 0.012237652830778625, 0.011329560521931974, 0.015282514783585829, 0.017073999258734963, 0.015751402426616903, 0.015598518859184712, 0.015369927485402567, 0.01594396006578761, 0.014931848731288264],
        "Cref": [0.016578371328500142, 0.01519492949780967, 0.008314211653128038, 0.012237652830778625, 0.011329560521931974, 0.015282514783585829, 0.017073999258734963, 0.06479249663883876, 0.17449259255126665, 0.17682112083446647, 0.21653782642309194, 0.24409487444912628],
    },
    "naver": {
        "baseline": [0.026535466343308493, 0.043699186842275264, 0.1091637047056099, 0.26788438341554316, 0.46942270929953334, 0.571301827637807, 0.5371129475808915, 0.5339065229528429, 0.5747029357196607, 0.5433898595996087, 0.4822155404687344, 0.45514819071073037],
        "floor": [0.026535466343308493, 0.013624620133663239, 0.010839588339364804, 0.011854571299371886, 0.012864818911651489, 0.015182811687894397, 0.01749119709271766, 0.015989610289790096, 0.017997490184067134, 0.01909746052484506, 0.021162288836076446, 0.022397308992923307],
        "Cref": [0.026535466343308493, 0.013624620133663239, 0.010839588339364804, 0.011854571299371886, 0.012864818911651489, 0.015182811687894397, 0.01749119709271766, 0.1583403691567614, 0.25113060615090343, 0.27180261502695613, 0.2961522945182981, 0.3296631443404199],
    },
    "DINOv3": {
        "baseline": [0.028061419654428127, 0.03723281990023723, 0.8416442362253422, 0.9360012704698906, 0.9486539607489823, 0.9314896120679399, 0.9404113062112893, 0.9414559726505077, 0.9325356727702198, 0.9300058708112987, 0.9305605942919077, 0.8522157092265887],
        "floor": [0.028061419654428127, 0.021471794619951495, 0.023721131535796264, 0.019373092589102255, 0.017711414187650976, 0.015312429216180267, 0.015525870121275985, 0.014515644619583141, 0.016405283766901212, 0.018909589709228538, 0.019979328858547377, 0.02528685315219093],
        "Cref": [0.028061419654428127, 0.021471794619951495, 0.023721131535796264, 0.019373092589102255, 0.017711414187650976, 0.015312429216180267, 0.015525870121275985, 0.4553550520783082, 0.4616016305073708, 0.4758900092424254, 0.5543148704736773, 0.4426402755470917],
    },
    "reg1_gap": {
        "baseline": [0.028885219906119627, -0.011069413089973215, 0.11068290083409153, 0.4308323959242171, 0.689023288054367, 0.7534770378763205, 0.7546736312236052, 0.6800232451832262, 0.6140928771605995, 0.5627457280720292, 0.5330771853332347, 0.44004141294977106],
        "floor": [0.028885219906119627, 0.01960807205278153, 0.01767489793363494, 0.014612943438481482, 0.014417140946612976, 0.017003321931897273, 0.011740003350952738, 0.009404632582739459, 0.012825332743719244, 0.01231742369452476, 0.011512894744813625, 0.01291190498057072],
        "Cref": [0.028885219906119627, 0.01960807205278153, 0.01767489793363494, 0.014612943438481482, 0.014417140946612976, 0.017003321931897273, 0.011740003350952738, 0.1035031725577133, 0.16042556506999336, 0.19547405362060394, 0.20199196992375926, 0.24457572022758525],
    },
    "rope_ape": {
        "baseline": [0.23404001348927642, 0.16566303569364965, 0.26573100817009093, 0.3753597807305771, 0.5897771336274423, 0.7737171521605842, 0.7612972714674181, 0.6208184515099673, 0.5568392761742282, 0.5234783900319877, 0.4695702927712563, 0.4261507060474893],
        "floor": [0.23404001348927642, 0.1930979520308264, 0.3486780133585031, 0.5363454254136164, 0.6337798686737818, 0.7517275369548175, 0.7285788640912101, 0.5255469759839224, 0.22755428891650062, 0.1352203741027112, 0.07856328140980308, 0.09115140091569521],
        "Cref": [0.23404001348927642, 0.1930979520308264, 0.3486780133585031, 0.5363454254136164, 0.6337798686737818, 0.7517275369548175, 0.7285788640912101, 0.5514959278255145, 0.40970114519904194, 0.3644586291507342, 0.3039335262388119, 0.2638544448379689],
    },
}
w8 = Path(W8_DIR).expanduser() if W8_DIR else None
if w8 and (w8 / "w8_partA_small.json").exists():
    dev = 0.0
    for key in RB0:
        c = json.load(open(w8 / f"w8_partA_{key}.json"))["conditions"]
        for cond in CONDITIONS:
            dev = max(dev, max(abs(a - b) for a, b in zip(c[cond]["ssdc"][:12], RB0[key][cond])))
    gate("typed RB-0 anchors equal the w8_partA files (all five models)", "anchors", dev == 0.0, f"max |delta| {dev:.1e}")
else:
    gate("typed RB-0 anchors equal the w8_partA files (all five models)", "anchors", True, "RB-0 folder not given; set RB1B_W8_DIR", skip=True)

## The worker
Same measurement as RB-1 and W5: `load_rope(input_size=224)`, W5's warm-up forward, then baseline, floor and
Cref with `evaluate_ssdc` (repack, 1,000 images, batch 256, seed 20260908, RPI, fp32). New in RB-1b: during
the warm-up forward on the real 256-image batch, the patch-embedding output is compared with a float64
computation of the same batch. A relative difference around 1e-4 or larger means a TF32 kernel; around
1e-7 means exact fp32. The worker also records which libcudnn file is loaded and exits with `os._exit(0)`.

In [ ]:
WORKER = r'''
import os, sys, json, time, hashlib, argparse
ap = argparse.ArgumentParser(); ap.add_argument("--cfg", required=True); args = ap.parse_args()
CFG = json.load(open(args.cfg))
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
torch.backends.cudnn.benchmark = False
if "cudnn_allow_tf32" in CFG["flags"]: torch.backends.cudnn.allow_tf32 = bool(CFG["flags"]["cudnn_allow_tf32"])
sys.path.insert(0, CFG["repo"])
from metrics.ssdc import evaluate_ssdc
from main.load_models import get_vit_blocks, get_block_attention, get_patch_embed_conv, load_rope
from main.prep_data import prep_data
from experiments.common import load_imagenet
TEST = CFG["test_mode"]; DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = {"setting": CFG["name"], "env_tag": CFG["env_tag"], "flags": CFG["flags"], "models": {}, "env": {}}
e = OUT["env"]
e["python"] = sys.executable; e["torch"] = torch.__version__; e["cuda"] = torch.version.cuda
e["cudnn"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
e["cudnn_allow_tf32"] = torch.backends.cudnn.allow_tf32; e["matmul_allow_tf32"] = torch.backends.cuda.matmul.allow_tf32
e["cudnn_deterministic"] = torch.backends.cudnn.deterministic
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    e["gpu"] = p.name; e["gpu_uuid"] = str(getattr(p, "uuid", "")); free, total = torch.cuda.mem_get_info()
    e["free_gb_at_start"] = free / 1e9

def find_rope(model):
    for n, m in model.named_modules():
        if callable(getattr(m, "get_embed", None)): return n, m
    return None, None
def identity_like(emb):
    s, c = emb.chunk(2, -1); return torch.cat([torch.zeros_like(s), torch.ones_like(c)], -1)
class RopeGlobal:
    def __init__(s, model, tf): s.name, s.mod = find_rope(model); s.tf = tf; s.handles = []
    def __enter__(s):
        s.orig = s.mod.get_embed; o, t = s.orig, s.tf
        s.mod.get_embed = (lambda _o=o, _t=t: (lambda *a, **k: _t(_o(*a, **k))))()
        s.handles = [1]; return s
    def __exit__(s, *e): s.mod.get_embed = s.orig; s.handles = []
class RopeComposite:
    def __init__(s, model, source, segs):
        s.blocks = get_vit_blocks(model, source); s.source = source; s.segs = [(list(b), t) for b, t in segs]; s.handles = []
    def __enter__(s):
        def mk(tf):
            def pre(mod, args, kw):
                r = kw.get("rope", None)
                if r is not None: kw = dict(kw); kw["rope"] = tf(r)
                return args, kw
            return pre
        for blocks, tf in s.segs:
            for i in blocks:
                s.handles.append(get_block_attention(s.blocks[i], s.source).register_forward_pre_hook(mk(tf), with_kwargs=True))
        return s
    def __exit__(s, *e):
        for h in s.handles: h.remove()
        s.handles = []
class NullCtx:
    handles = []
    def __enter__(s): return s
    def __exit__(s, *e): return False

if TEST:
    from PIL import Image as _PIL
    class _Fake:
        def __iter__(self):
            rng = np.random.default_rng(1)
            for i in range(400):
                yield {"image": _PIL.fromarray(rng.integers(0, 256, size=(256, 256, 3), dtype=np.uint8)), "label": i % 10}
    DATASET = _Fake()
else:
    DATASET = load_imagenet(split="validation", streaming=True, dataset_id=CFG["dataset"], fallback_id=None)

def load(key):
    if TEST:
        import timm
        torch.manual_seed(1234 + len(key))
        nh = 6 if key == "small" else 12
        m = timm.create_model(CFG["model_names"][key].split(".")[0], pretrained=False, dynamic_img_size=True,
                              embed_dim=96 if key == "small" else 192, num_heads=nh,
                              num_classes=0 if key == "DINOv3" else 10).to(DEVICE).eval()
        with torch.no_grad():
            for blk in m.blocks:
                for g in ("gamma_1", "gamma_2"):
                    t = getattr(blk, g, None)
                    if isinstance(t, torch.Tensor): t.fill_(1.0)
        cfg = timm.data.resolve_model_data_config(m); cfg["input_size"] = (3, CFG["input_size"], CFG["input_size"])
        return m, timm.data.create_transform(**cfg, is_training=False), "timm"
    return load_rope(model_name=CFG["model_names"][key], device=DEVICE, half=False, input_size=CFG["input_size"])

N, B, SEED = CFG["n_images"], CFG["batch_size"], CFG["seed"]
for key in CFG["models"]:
    tm = time.time()
    model, processor, source = load(key)
    model.eval().float()
    conv = get_patch_embed_conv(model, source)
    rec = {"conditions": {}}
    # W5 forward_grid warm-up, on the real first batch; the patch-embedding output of that batch is kept
    grab = {}
    h = conv.register_forward_hook(lambda m, i, o: grab.update(x=i[0].detach(), y=o.detach()))
    dl = prep_data(DATASET, processor, source, corruption_type=None, number_images=B, batch_size=B, half=False, num_workers=0)
    with torch.no_grad():
        for im, _ in dl:
            model(im.to(DEVICE)); break
    h.remove()
    with torch.no_grad():
        y64 = F.conv2d(grab["x"].double(), conv.weight.double(), None if conv.bias is None else conv.bias.double(),
                       conv.stride, conv.padding, conv.dilation, conv.groups)
        rec["first_layer_rel_diff_vs_float64"] = float((grab["y"].double() - y64).abs().max() / y64.abs().max())
        rec["first_layer_batch"] = int(grab["x"].shape[0])
    torch.save(grab["y"][:2].float().cpu().clone(), CFG["out_prefix"] + f"_{key}_patch_b{rec['first_layer_batch']}.pt")
    del grab, y64
    def measure(ctx):
        torch.manual_seed(SEED); np.random.seed(SEED % (2 ** 31))
        with ctx:
            scores, _ = evaluate_ssdc(model, processor, DATASET, source, RPI=True, number_images=N, batch_size=B,
                                      metric=CFG["metric"], half=False, num_workers=0)
        return [float(v) for v in scores]
    rec["conditions"]["baseline"] = measure(NullCtx())
    rec["conditions"]["floor"] = measure(RopeGlobal(model, identity_like))
    rec["conditions"]["Cref"] = measure(RopeComposite(model, source, [(CFG["window"], identity_like)]))
    dl = prep_data(DATASET, processor, source, corruption_type=None, number_images=N, batch_size=B, half=False, num_workers=0)
    hh = hashlib.sha256(); n_seen = 0
    for im, lb in dl:
        hh.update(im.contiguous().numpy().tobytes()); hh.update(torch.as_tensor(lb).numpy().tobytes()); n_seen += int(im.shape[0])
    rec["image_sha256"] = hh.hexdigest(); rec["n_images_hashed"] = n_seen
    hw = hashlib.sha256()
    for k_, v_ in sorted(model.state_dict().items()):
        hw.update(k_.encode()); hw.update(v_.detach().float().cpu().contiguous().numpy().tobytes())
    rec["weights_sha256"] = hw.hexdigest()
    rec["runtime_s"] = round(time.time() - tm, 1)
    OUT["models"][key] = rec
    print(f"  {CFG['name']:20s} {key:8s} first layer vs float64 {rec['first_layer_rel_diff_vs_float64']:.1e} | "
          f"b4 baseline {rec['conditions']['baseline'][4]:.9f} | Cref b11 {rec['conditions']['Cref'][11]:.9f} | {rec['runtime_s']}s", flush=True)
    del model
    if DEVICE == "cuda": torch.cuda.empty_cache()
e["libcudnn_loaded"] = sorted({l.split()[-1] for l in open("/proc/self/maps") if "libcudnn" in l and l.split()[-1].startswith("/")})
json.dump(OUT, open(CFG["out_prefix"] + ".json", "w"), indent=1)
print("WORKER DONE", flush=True)
sys.stdout.flush(); sys.stderr.flush()
os._exit(0)   # skip interpreter shutdown, which crashed harmlessly in W5 and RB-1
'''
WORKER_PATH = OUT_DIR / "rb1b_worker.py"
WORKER_PATH.write_text(WORKER)
print("worker written:", WORKER_PATH)

## Run the settings
Each setting runs in a fresh process with that setting's Python environment. A setting counts as complete
when its result file exists and contains every requested model; the exit code is only recorded.

In [ ]:
def run_setting(name, env_tag, flags, models):
    prefix = str(OUT_DIR / f"rb1b_{name}")
    done = Path(prefix + ".json")
    if done.exists() and name not in ONLY:
        print(f"{name}: already done, loading"); return json.loads(done.read_text())
    cfg = {"name": name, "env_tag": env_tag, "flags": flags, "models": models, "repo": str(repo), "dataset": SSDC_DATASET,
           "n_images": N_IMAGES, "batch_size": BATCH_SIZE, "seed": TORCH_SEED, "metric": METRIC,
           "input_size": INPUT_SIZE, "window": WINDOW, "model_names": MODEL_NAMES, "test_mode": TEST_MODE,
           "out_prefix": prefix}
    cfg_path = Path(prefix + "_cfg.json"); cfg_path.write_text(json.dumps(cfg))
    done.unlink(missing_ok=True)
    t = time.time()
    r = subprocess.run([PYTHONS[env_tag], str(WORKER_PATH), "--cfg", str(cfg_path)], env=dict(os.environ),
                       capture_output=True, text=True, timeout=7200)
    Path(prefix + "_stdout.txt").write_text(r.stdout + f"\n--- exit code {r.returncode} ---\n--- stderr ---\n" + r.stderr)
    print(r.stdout.strip())
    out = json.loads(done.read_text()) if done.exists() else None
    ok = out is not None and set(out["models"]) == set(models)
    if not ok:
        print(r.stderr[-3000:])
    gate("setting completed (result saved for every model)", name, ok, f"{time.time() - t:.0f}s, exit code {r.returncode}")
    return out if ok else None

RUNS = {}
if PRE_OK:
    for name, env_tag, flags, models in SETTINGS:
        if ONLY and name not in ONLY and not (OUT_DIR / f"rb1b_{name}.json").exists():
            continue
        print("\n" + "=" * 78 + f"\n{name}  env={env_tag}  flags={flags}  models={models}\n" + "=" * 78, flush=True)
        try:
            r = run_setting(name, env_tag, flags, models)
            if r: RUNS[name] = r
        except Exception as ex:
            traceback.print_exc(); gate("setting completed (result saved for every model)", name, False, repr(ex)[:200])
else:
    print("pre-flight failed: nothing measured")
for name, R in RUNS.items():
    e = R["env"]
    print(f"{name:20s} cuDNN {e['cudnn']} | loaded {[Path(x).name for x in e.get('libcudnn_loaded', [])]} | "
          f"{e.get('gpu_uuid', '')} | conv TF32 allowed {e['cudnn_allow_tf32']}")
    if not TEST_MODE:
        v = e["cudnn"] or 0
        gate("worker ran with the intended cuDNN", name, (v // 10000, (v // 100) % 100) == EXPECT[R["env_tag"]], str(v))

## Analysis

In [ ]:
pd.set_option("display.width", 250)
def maxdev(a, b): return max(abs(x - y) for x, y in zip(a, b))
def fmt(d): return "-" if d is None else ("0" if d == 0.0 else f"{d:.1e}")
rows = []
for name, R in RUNS.items():
    for key, rec in R["models"].items():
        row = {"setting": name, "model": key, "first_layer_vs_float64": rec["first_layer_rel_diff_vs_float64"]}
        if not TEST_MODE:
            row["vs_W5"] = max(maxdev(rec["conditions"][c], W5[key][c]) for c in CONDITIONS)
            row["vs_RB0"] = max(maxdev(rec["conditions"][c], RB0[key][c]) for c in CONDITIONS)
            row["b0_vs_W5"] = rec["conditions"]["baseline"][0] - W5[key]["baseline"][0]
        rows.append(row)
DF = pd.DataFrame(rows)
if len(DF):
    show = DF.copy()
    for c in ("vs_W5", "vs_RB0"):
        if c in show: show[c] = show[c].map(fmt)
    if "b0_vs_W5" in show: show["b0_vs_W5"] = show["b0_vs_W5"].map(lambda x: f"{x:+.1e}")
    show["first_layer_vs_float64"] = show["first_layer_vs_float64"].map(lambda x: f"{x:.1e}")
    print("worst |difference| over 12 block inputs and three conditions ('0' = bit-identical)\n")
    print(show.to_string(index=False))
    print("\nfirst_layer_vs_float64: ~1e-7 = exact fp32 kernel; ~1e-4 or more = TF32 kernel (on the real 256-image batch)")

for key in MODEL_NAMES:
    img = {n: R["models"][key]["image_sha256"] for n, R in RUNS.items() if key in R["models"]}
    wts = {n: R["models"][key]["weights_sha256"] for n, R in RUNS.items() if key in R["models"]}
    if img:
        gate("images identical in every setting and both environments", key, len(set(img.values())) == 1)
        gate("weights identical in every setting and both environments", key, len(set(wts.values())) == 1)

def worst(name, key, ref):
    r = DF[(DF.setting == name) & (DF.model == key)]
    return float(r[ref].iloc[0]) if len(r) and ref in r else None

## Verdict
Rules fixed before the run.

* **Case 1 - exact.** cuDNN 9.10.2 reproduces September bit for bit on all five models. Cause found.
  Next: run everything in the clean copy, starting with an unchanged re-run of RB-0, then RB-2.
* **Case 2 - ViT-S fixed.** With 9.10.2, ViT-S comes within 1.1e-6 of September (the level of the other
  models' drift), though not every model is bit-identical. cuDNN caused the large ViT-S drift; the tiny rest
  is the new driver or card. Next: same as case 1.
* **Case 3 - partial.** 9.10.2 moves ViT-S closer to September but not within 1.1e-6. Next: we decide together
  before any further run.
* **Case 4 - not cuDNN.** 9.10.2 leaves ViT-S's drift essentially unchanged (within 10%). Next: keep
  `env_paper5`, gate everything against RB-0, add one appendix line, and start RB-2.

In [ ]:
VERDICT = {}
if not TEST_MODE and RUNS:
    ctrl = RUNS.get("cudnn926_default")
    if ctrl:
        d = max(worst("cudnn926_default", k, "vs_RB0") for k in ctrl["models"])
        gate("control: env_paper5 (cuDNN 9.26) still reproduces RB-0 on all five models", "control", d == 0.0, fmt(d))
    a, b = RUNS.get("cudnn910_default_a"), RUNS.get("cudnn910_default_b")
    if a and b:
        rep = max(maxdev(a["models"][k]["conditions"][c], b["models"][k]["conditions"][c]) for k in b["models"] for c in CONDITIONS)
        gate("the cuDNN 9.10.2 environment is repeatable (default_a = default_b)", "cudnn910", rep == 0.0, fmt(rep))
    if a:
        d910 = {k: worst("cudnn910_default_a", k, "vs_W5") for k in a["models"]}
        d926 = {k: worst("cudnn926_default", k, "vs_W5") for k in a["models"]} if ctrl else {}
        VERDICT["vs_September_with_cudnn910"] = d910
        VERDICT["vs_September_with_cudnn926"] = d926
        print("worst difference from September, per model:")
        for k in a["models"]:
            print(f"  {k:9s} cuDNN 9.10.2: {fmt(d910[k]):8s}  cuDNN 9.26: {fmt(d926.get(k))}")
        s910, s926 = d910.get("small"), d926.get("small")
        if all(v == 0.0 for v in d910.values()) and len(d910) == 5:
            case = ("1", "cuDNN 9.10.2 reproduces September bit for bit on all five models. The drift came from env_paper5 "
                         "loading cuDNN 9.26 instead of the 9.10.2 that PyTorch 2.10 is built for.")
        elif s910 is not None and s910 <= 1.1e-6:
            case = ("2", f"With cuDNN 9.10.2, ViT-S comes back to within {fmt(s910)} of September (it was {fmt(s926)} with 9.26). "
                         "cuDNN 9.26 caused ViT-S's large drift; what is left is at the level every model showed after the "
                         "driver and card change.")
        elif s910 is not None and s926 is not None and s910 < 0.9 * s926:
            case = ("3", f"cuDNN 9.10.2 moves ViT-S closer to September ({fmt(s926)} -> {fmt(s910)}) but not within 1.1e-6.")
        else:
            case = ("4", f"cuDNN 9.10.2 leaves ViT-S's drift essentially unchanged ({fmt(s926)} -> {fmt(s910)}). cuDNN is not the cause; "
                         "what remains is the driver and physical-card change, which cannot be undone.")
        VERDICT["case"], VERDICT["explanation"] = case
        VERDICT["next"] = {"1": "Use the clean cuDNN 9.10.2 environment from now on. Re-run RB-0 there unchanged (~2 h), then RB-2.",
                           "2": "Use the clean cuDNN 9.10.2 environment from now on. Re-run RB-0 there unchanged (~2 h), then RB-2.",
                           "3": "Stop here and send the results; we decide together before any further run.",
                           "4": "Keep env_paper5. Gate every new run against RB-0 and start RB-2."}[case[0]]
        tf = RUNS.get("cudnn910_tf32_off")
        if tf:
            VERDICT["tf32_off_vs_September_small"] = worst("cudnn910_tf32_off", "small", "vs_W5")
            VERDICT["first_layer_small"] = {n: R["models"]["small"]["first_layer_rel_diff_vs_float64"]
                                            for n, R in RUNS.items() if "small" in R["models"]}
        print(f"\nCASE {case[0]}: {case[1]}\nNEXT: {VERDICT['next']}")
        if "first_layer_small" in VERDICT:
            print("\nViT-S first layer vs float64 (real batch):", {k: f"{v:.1e}" for k, v in VERDICT["first_layer_small"].items()})
            print("ViT-S with cuDNN 9.10.2 and TF32 off vs September:", fmt(VERDICT["tf32_off_vs_September_small"]))
else:
    print("test mode or nothing measured: verdict skipped")

## Gate report and export

In [ ]:
_all = []
for f in sorted(OUT_DIR.glob("rb1b_gates_session_*.json")):
    _all += json.loads(f.read_text())
_seen = {}
for g in _all: _seen[(g["scope"], g["gate"])] = g
gdf = pd.DataFrame(list(_seen.values()))
print(f"GATE REPORT ({len(set(g['session'] for g in _all))} session(s), latest result per gate)\n")
print(gdf.to_string(index=False) if len(gdf) else "(none)")
n_fail = int((gdf.status == "FAIL").sum()) if len(gdf) else 0
print(f"\n{len(gdf)} gates: {int((gdf.status == 'PASS').sum())} pass, {n_fail} fail, {int((gdf.status == 'SKIP').sum())} skipped")
print("ALL GATES PASSED" if n_fail == 0 else f"{n_fail} GATE(S) FAILED: send the whole rb1b_output folder")
export = {"notebook": "RB-1b", "generated_utc": RUN_STAMP, "test_mode": TEST_MODE, "repo_commit": REPO_COMMIT, "driver": DRIVER,
          "preflight": PRE, "settings": {n: {"env_tag": R["env_tag"], "flags": R["flags"], "env": R["env"]} for n, R in RUNS.items()},
          "comparison": DF.to_dict("records") if len(DF) else [], "verdict": VERDICT,
          "gates": gdf.to_dict("records") if len(gdf) else []}
(OUT_DIR / f"rb1b_all_{RUN_STAMP}.json").write_text(json.dumps(export, indent=1, default=str))
if len(DF): DF.to_csv(OUT_DIR / "rb1b_comparison.csv", index=False)
gdf.to_csv(OUT_DIR / "rb1b_gate_report.csv", index=False)
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print(f"\narchive: {archive} ({os.path.getsize(archive):,} bytes)")